# 05 — One-slot genes

This notebook walks through the one-slot symbolic representation end to end. The core invariant is simple:

> **One operation invocation appends exactly one physical gene.**

If the operation logically emits many values, those values live inside a nested object array within that one gene. We will use a synthetic ARC rule where the output is the input rotated clockwise by 90 degrees, then follow the representation through generation, component addressing, GP↔SP solving, caching, compiler materialization, and `crawl_synth_v2`.

The old multi-gene registry/search path remains available as legacy. Everything in this notebook uses the bundled path unless explicitly labeled as the temporary compiler bridge.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ops import (
    BUNDLED_OP_REGISTRY,
    ComponentPairEvaluationCache,
    apply_bundled_operation,
    bundle_manifest,
    component_dims,
    component_dtype_strings,
    component_gene,
    component_refs_for_gene,
    crawl_synth_v2,
    init_env_bundled,
    materialize_legacy_environment,
    solve_bundled_frontier,
)
from ops.kelschinator import Kelschinator

print('repo:', ROOT)
print('bundled operations:', len(BUNDLED_OP_REGISTRY))

## 1. Build an example transformation

The training rule is **clockwise rotation**. The examples deliberately use rectangular grids so the transform also changes shape. This gives the shape partition, composite partition, and matrix operation meaningful work to do.

In [ ]:
train = [
    {
        'input': np.array([[1, 0, 2], [0, 3, 0]], dtype=np.int64),
        'output': np.array([[0, 1], [3, 0], [0, 2]], dtype=np.int64),
    },
    {
        'input': np.array([[4, 0, 0], [5, 6, 0]], dtype=np.int64),
        'output': np.array([[5, 4], [6, 0], [0, 0]], dtype=np.int64),
    },
]

X_test = np.array([[7, 0, 8], [0, 9, 0]], dtype=np.int64)
Y_test = np.rot90(X_test, k=-1).copy()

def show_pair(x, y, title):
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    for ax, grid, label in zip(axes, [x, y], ['input', 'output']):
        ax.imshow(grid, interpolation='nearest', vmin=0, vmax=9)
        ax.set_title(label)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

for i, sample in enumerate(train):
    show_pair(sample['input'], sample['output'], f'train {i}')
show_pair(X_test, Y_test, 'unseen test')

## 2. Initialize the bundled GP / SP environments

`init_env_bundled` starts each side with exactly three physical genes:

1. raw matrix
2. one shape bundle `[[h, w]]`
3. one composite bundle `[[color, mask], ...]`

The number of colors no longer changes the physical gene count.

In [ ]:
GP_meta, GP_X, SP_meta, SP_X, ST = init_env_bundled(train)

print('physical GP genes:', len(GP_X))
print('physical SP genes:', len(SP_X))
print('GP ops:', GP_meta.op)
print('SP ops:', SP_meta.op)

assert len(GP_X) == 3
assert len(SP_X) == 3

In [ ]:
print('Shape bundle, train sample 0:')
print(GP_X[1, 0])

print('\nComposite bundle, train sample 0:')
for row in GP_X[2, 0]:
    color, mask = row
    print('color =', int(color))
    print(mask.astype(int))

print('\nPhysical GP count is still:', len(GP_X))

## 3. Address semantic components without creating genes

A physical bundle exposes virtual leaves through `GeneComponentRef(gene_id, path)`. A component path is an address into the nested object value; extracting it produces a temporary sample-axis view for evaluation only.

In [ ]:
rows = []
for gidx in range(len(GP_X)):
    for ref in component_refs_for_gene(GP_meta, GP_X, gidx):
        rows.append({
            'physical_gidx': gidx,
            'stable_gene_id': ref.gene_id,
            'path': ref.path,
            'dims': component_dims(GP_meta, GP_X, ref),
            'dtypes': component_dtype_strings(GP_meta, GP_X, ref),
            'sample0_shape': np.asarray(component_gene(GP_meta, GP_X, ref)[0]).shape,
        })

component_table = pd.DataFrame(rows)
display(component_table)

before = len(GP_X)
example_ref = component_refs_for_gene(GP_meta, GP_X, 1)[0]
example_component = component_gene(GP_meta, GP_X, example_ref)
print('virtual component values:', [int(v) for v in example_component])
print('physical gene count unchanged:', before, '->', len(GP_X))

## 4. Apply a real operation: still one new physical slot

The desired rule is `mat2_cwrotate`. Applying it to raw GP gene 0 creates one physical gene, even though the framework is capable of mapping an operation across multiple compatible components inside a source bundle.

In [ ]:
before = len(GP_X)
rotate_gidx = apply_bundled_operation(
    GP_meta,
    GP_X,
    'mat2_cwrotate',
    0,
)

print('new gidx:', rotate_gidx)
print('physical GP count:', before, '->', len(GP_X))
print('operation:', GP_meta.op[rotate_gidx])
print('sample 0 bundled value:')
print(GP_X[rotate_gidx, 0])

assert len(GP_X) == before + 1
assert np.array_equal(GP_X[rotate_gidx, 0][0, 0], train[0]['output'])

In [ ]:
manifest = bundle_manifest(GP_meta, rotate_gidx)
print(json.dumps(manifest, indent=2))

print('\nThe manifest records:')
print('- which source component path each logical application consumed')
print('- which output component path it produced')
print('- dimensionality and dtype metadata for every nested component')

## 5. Solve SP/ST by comparing component views

The solver never explodes bundles into persistent genes. It enumerates component refs, extracts temporary views, and records evaluated GP/SP component pairs in a sparse stable cache keyed by `(gene_id, path)`.

In [ ]:
cache = ComponentPairEvaluationCache()

print('ST solved before component matching?', ST.solved)
solutions = solve_bundled_frontier(
    GP_meta,
    GP_X,
    SP_meta,
    SP_X,
    ST,
    cache,
)

print('new component solutions:', len(solutions))
for solution in solutions:
    print(
        'SP', solution.sp_ref,
        '<-', solution.rule,
        'GP', solution.gp_ref,
    )

print('ST solved after matching?', ST.solved)
print('cached 0D pairs:', len(cache.evaluated.get(0, set())))
print('cached 2D pairs:', len(cache.evaluated.get(2, set())))

assert ST.solved

## 6. Compile the solved bundled proof

Search/storage stays one-slot. Only after the ST is solved do we create an **ephemeral legacy view**. Every referenced component becomes a temporary legacy gene with lineage reconstructed from the bundle manifest. This is a compiler bridge so the existing Kelschinator can execute the proof on unseen inputs; it is not fed back into search.

In [ ]:
(
    legacy_GP_meta,
    legacy_GP_X,
    legacy_SP_meta,
    legacy_SP_X,
    legacy_ST,
) = materialize_legacy_environment(
    GP_meta,
    GP_X,
    SP_meta,
    SP_X,
    ST,
)

print('bundled physical GP genes:', len(GP_X))
print('temporary legacy GP components:', len(legacy_GP_X))
print('bundled physical SP genes:', len(SP_X))
print('temporary legacy SP components:', len(legacy_SP_X))
print('legacy bridge ST solved:', legacy_ST.solved)

In [ ]:
kelschinator = Kelschinator()
fit_ok = kelschinator.fit(legacy_ST)
print('fit:', fit_ok)
print('last error:', kelschinator.last_error_)
print('pipeline:')
for line in kelschinator.pipeline_:
    print(' ', line)

y_hat = kelschinator.transform(X_test)
print('\nexact unseen test match:', np.array_equal(y_hat, Y_test))
show_pair(X_test, y_hat, 'Kelschinator prediction')

assert fit_ok
assert np.array_equal(y_hat, Y_test)

## 7. Run the same idea through `crawl_synth_v2`

For a fully reproducible demo, the next cell writes the same rotation task into a temporary ARC-style training directory and lets `crawl_synth_v2` discover it. The crawler keeps the v1 scheduling and persistent Grammar-UCT behavior, but each accepted generation is now one physical bundled gene.

In [ ]:
task_payload = {
    'train': [
        {'input': s['input'].tolist(), 'output': s['output'].tolist()}
        for s in train
    ],
    'test': [
        {'input': X_test.tolist(), 'output': Y_test.tolist()}
    ],
}

with tempfile.TemporaryDirectory() as tmp:
    tmp_root = Path(tmp)
    task_id = 'oneslot_rotation'
    task_path = tmp_root / 'data' / 'training' / f'{task_id}.json'
    task_path.parent.mkdir(parents=True, exist_ok=True)
    task_path.write_text(json.dumps(task_payload))

    crawl_result = crawl_synth_v2(
        first_tasks=[task_id],
        max_GP=30,
        max_SP=15,
        prune_size_GP=0,
        max_total_generations=25,
        max_task_generations=20,
        operation_selection='argmax',
        source_probe_attempts=32,
        rng=0,
        data_root=tmp_root,
        verbosity=2,
        show_success_plots=False,
    )

print('\nSolved IDs:', crawl_result.solved_task_ids)
display(pd.DataFrame([
    {
        'task_id': a.task_id,
        'solved_exactly': a.solved_exactly,
        'one_slot_gp_generations': a.generated_genes,
        'iterations': a.iterations,
        'final_physical_gp': a.final_gp_len,
        'final_physical_sp': a.final_sp_len,
        'seconds': a.elapsed_seconds,
    }
    for a in crawl_result.attempts
]))

## 8. Representation summary

The important separation is now:

```text
physical search tree
    one operation -> one gene
              |
              +-- nested semantic components
                       |
                       +-- virtual GeneComponentRef leaves
                                  |
                                  +-- GP/SP solver + sparse cache
                                             |
                                             +-- solved ST
                                                    |
                                                    +-- temporary compiler flattening
                                                               |
                                                               +-- Kelschinator
```

Search complexity, depth, pruning, and UCT reward now operate on physical one-slot genes. Semantic comparison can still reason about every nested value independently.